# 05 - Garfield (Phase B3) and Cross-Platform Generalization (Phase C)

**Scope correction:** this notebook originally also covered STAGATE. That
turned out to be unnecessary -- STAGATE's "blocked on Windows" status was a
stale claim (PyG's wheel index now covers torch 2.11.0+cu128), so it now runs
locally: `src/eval/run_stagate_dlpfc.py`, full 12-slice x 5-seed results
already in `outputs/logs/stagate_dlpfc_results.json`, no Colab needed. This
notebook now covers the two things that genuinely still need Colab (or WSL):

1. **Phase B3 - Garfield.** Verified blocked, not stale: it depends on
   `pybedtools` -> `pysam` -> `htslib`, which has no Windows wheel (confirmed by
   attempting the install directly and watching `pysam`'s build fail).
2. **Phase C - generalization.** Every number in this project so far comes from
   one tissue on one platform (DLPFC, 10x Visium). A literature check
   (checked directly against the GraphST paper, PMC9977836) found the original
   plan's platform ordering was backwards: GraphST reports **no ARI** for
   either mouse olfactory bulb (Stereo-seq) or Slide-seqV2 hippocampus --
   both are qualitative (marker-gene / atlas comparison) only. **Human breast
   cancer (10x Visium) is the only one of the three platforms with a
   published, directly comparable GraphST ARI** (0.54-0.57 vs. pathologist
   annotation, 20 regions), so it leads here.

**Design rule, unchanged: this notebook CLONES the repo rather than inlining
model code.** The original `04_colab_scaleup.ipynb` pasted a copy of the
Phase 0 `EmbeddedMemoryLayer` inline; that copy is now many stages stale and
would silently benchmark a model nobody uses.


## 0. Environment


In [ ]:
!nvidia-smi


In [ ]:
!git clone -q https://github.com/AnkitDash-code/Memory_RECOMB-27.git repo
%cd repo/recomb2027
!pip install -q scanpy squidpy anndata scikit-learn scikit-misc pot GraphST entmax

import sys
sys.path.insert(0, '.')
import torch
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', DEVICE)


## 1. Phase B3 - Garfield

Confirmed on PyPI as v1.0.0, the spatial-omics package (Weige Zhou,
github.com/zhou-1314/Garfield), not a name collision. Its README documents
parameters but ships no concrete quickstart, so the API is discovered here
at runtime rather than guessed -- a confident-looking call against an
unverified API is worse than writing none.


In [ ]:
!pip install -q garfield


In [ ]:
import garfield, inspect
print('version:', getattr(garfield, '__version__', 'unknown'))
print('top-level:', [n for n in dir(garfield) if not n.startswith('_')])
for name in [n for n in dir(garfield) if not n.startswith('_')][:15]:
    obj = getattr(garfield, name)
    if inspect.isclass(obj) or inspect.isfunction(obj):
        try:
            print(f'\n{name}{inspect.signature(obj)}')
        except (ValueError, TypeError):
            print(f'\n{name} (signature unavailable)')


Once the API shape is known: run Garfield on DLPFC 151673 (5 seeds), score
the embedding with `src/eval/clustering.py::cluster_embedding` (the identical
protocol every other method here uses), and extend
`src/eval/significance_test_stagate.py`'s pattern to a fourth comparator if
the number looks credible against the 0.633 literature reference.


In [ ]:
# from src.eval.clustering import cluster_embedding
# TODO: fill in once the API discovery cell above shows the actual
# construct/train/embedding-retrieval calls.


## 2. Phase C - cross-platform generalization

Leading with the platform that has a real literature number to compare
against, per the corrected priority:

| dataset | platform | why | GraphST ARI in literature |
|---|---|---|---|
| **Human breast cancer** | **10x Visium** | **pathologist-annotated, 20 regions -- the only one of the three with a published GraphST ARI** | **0.54-0.57** |
| Mouse olfactory bulb | Stereo-seq | different tissue, same general platform family | none reported (qualitative only) |
| Mouse hippocampus | Slide-seqV2 | bead-based, not spot-array -- the strongest platform-shift test | none reported (qualitative only) |

**Methodological guardrail, carried over from DLPFC:** do not retune the
architecture per dataset to chase numbers. Re-running an *already-validated*
selection procedure is fine; introducing new per-dataset architectural changes
is the same leakage as tuning on the test set, spread across datasets instead
of slices. Use the DLPFC-validated defaults (`memory_slots=16`, `n_hops=4`,
`lambda_usage=0.02`, expression-weighted adjacency) as-is.


### 2a. Human breast cancer (10x Visium) -- run this first

The dataset GraphST's own paper reports 0.54-0.57 ARI on (20-region
pathologist annotation). This is the one result in Phase C directly
comparable to a published number, so it should be the first thing that runs
and the first thing reported, even if the other two platforms take longer to
source or annotate.


In [ ]:
# TODO: source the same 10x Visium human breast cancer dataset GraphST's paper
# uses (their repo/tutorial names the exact sample) and the 20-region
# pathologist annotation, so the ARI is comparable to the published 0.54-0.57.
import squidpy as sq
# breast_adata = sq.datasets.???  # confirm exact dataset id against GraphST's tutorial


In [ ]:
import numpy as np
from sklearn.metrics import adjusted_rand_score

from src.data.preprocess import preprocess_hvg
from src.eval.clustering import cluster_embedding, consensus_cluster
from src.models.run_graphst import run_graphst
from src.models.train_spatial_address import train_spatial_address_model


def evaluate_dataset(raw, n_clusters, label_key=None, seeds=(0, 1, 2, 3, 4),
                     coord_type='grid'):
    """coord_type='grid' for Visium (hex/square grid); 'generic' for
    non-Visium bead-based platforms (Slide-seqV2), where the grid assumption
    does not hold."""
    adata = preprocess_hvg(raw.copy(), coord_type=coord_type)
    truth = adata.obs[label_key] if label_key else None
    mask = truth.notna().to_numpy() if truth is not None else None

    out = {}
    for name in ['ours', 'graphst']:
        labels_per_seed, aris = [], []
        for seed in seeds:
            if name == 'ours':
                _, tr, _ = train_spatial_address_model(adata.copy(), seed=seed,
                                                       device=DEVICE, verbose=False)
                emb, coords = tr.obsm['X_spatial_address'], tr.obsm['spatial']
            else:
                g = run_graphst(raw.copy(), n_clusters=n_clusters, device=DEVICE,
                                random_seed=seed, cluster=False)
                emb, coords = g.obsm['emb'], g.obsm['spatial']
            labels = cluster_embedding(emb, n_clusters, coords=coords, refine=True)
            labels_per_seed.append(labels)
            if truth is not None:
                aris.append(adjusted_rand_score(truth[mask], np.asarray(labels)[mask]))
        cons = consensus_cluster(labels_per_seed, n_clusters)
        out[name] = {
            'per_seed': aris,
            'mean': float(np.mean(aris)) if aris else None,
            'std': float(np.std(aris)) if aris else None,
            'consensus': (float(adjusted_rand_score(truth[mask], np.asarray(cons)[mask]))
                          if truth is not None else None),
        }
    return out


### 2b. Mouse olfactory bulb (Stereo-seq) and mouse hippocampus (Slide-seqV2)

Neither has a published GraphST ARI (confirmed above), so neither can produce
a number directly comparable to the literature the way breast cancer can.
Report these the way the source papers do: unsupervised metrics (silhouette,
spatial coherence) plus qualitative marker-gene / laminar-structure agreement
-- not a manufactured supervised ARI against an annotation GraphST itself
never scored against.

**Slide-seqV2 caveat:** squidpy's `sq.datasets.slideseqv2()` ships cell-type
labels, not spatial-domain labels. ARI against those measures a different
task than DLPFC layers -- a spatial-domain method is not supposed to recover
cell types. STAGATE reportedly distributes an annotated Slide-seqV2 mouse OB;
if used, note explicitly that it is STAGATE's own annotation, not a shared
community standard, when reporting any resulting ARI.


In [ ]:
import squidpy as sq

slide = sq.datasets.slideseqv2()
print(slide)
print('sparsity:', 1 - slide.X.nnz / (slide.X.shape[0] * slide.X.shape[1]))
print('obs columns:', list(slide.obs.columns))


## 3. Reporting

Bring results back into the repo and score them with the same statistical
machinery as DLPFC -- `src/eval/significance_test.py` and the
`significance_test_stagate.py` pattern report paired Wilcoxon plus
rank-biserial effect size plus bootstrap CI, because at small n a p-value
alone cannot distinguish 'no effect' from 'underpowered'. Report every
platform whether it wins, ties, or loses; the pattern is the finding.
